# Train Distance Pipeline

This notebook geocodes every unique HDB resale flat address and finds its nearest public rail station (MRT or LRT) using the [OneMap Singapore API](https://www.onemap.gov.sg/apidocs/).

## Inputs
- `../merged_data/merged_hdb_resale_with_macro.csv` — the enriched HDB resale dataset
- `../.env` — OneMap credentials (fill in before running):
  ```
  ONEMAP_EMAIL=your_email
  ONEMAP_PASSWORD=your_password
  ```

## How to Run
1. Fill in your OneMap credentials in `../.env`
2. Run cells top to bottom — **Phase 3 and Phase 4 are resume-safe**: if the notebook is interrupted, re-running will skip already-cached addresses.

## Output Files
| File | Description |
|------|-------------|
| `../merged_data/hdb_with_train_distances.csv` | Full dataset enriched with lat, lon, and nearest train columns |
| `../data/geocode_cache.json` | Cache of geocoded addresses (auto-created) |
| `../data/train_cache.json` | Cache of nearest train results (auto-created) |
| `../data/failed_geocodes.csv` | Addresses that could not be geocoded |

In [1]:
import pandas as pd
import json
import os
import time
import math
import requests
from tqdm.auto import tqdm
from dotenv import load_dotenv

print("Libraries imported successfully!")

Libraries imported successfully!


C:\Users\joelc\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Phase 1 — Extract Unique Addresses

Load the merged HDB dataset and extract all unique block + street_name combinations. These are the addresses we need to geocode — working with unique addresses avoids redundant API calls for the same location.

In [2]:
INPUT_CSV = '../merged_data/merged_hdb_resale_with_macro.csv'

print(f"Loading dataset from {INPUT_CSV}...")
df = pd.read_csv(INPUT_CSV)
print(f"Dataset shape: {df.shape}")

unique_addresses = df[['block', 'street_name']].drop_duplicates().reset_index(drop=True)
print(f"\nUnique addresses found: {len(unique_addresses):,}")
print("\nSample addresses:")
print(unique_addresses.head(10))

Loading dataset from ../merged_data/merged_hdb_resale_with_macro.csv...
Dataset shape: (197432, 19)

Unique addresses found: 9,660

Sample addresses:
  block        street_name
0   314   ANG MO KIO AVE 3
1   156   ANG MO KIO AVE 4
2   463  ANG MO KIO AVE 10
3   503   ANG MO KIO AVE 5
4   445  ANG MO KIO AVE 10
5   565   ANG MO KIO AVE 3
6   178   ANG MO KIO AVE 4
7   182   ANG MO KIO AVE 5
8   214   ANG MO KIO AVE 3
9   504   ANG MO KIO AVE 8


## Phase 2 — OneMap Authentication

Load credentials from `.env` and authenticate with the OneMap API. A shared `requests.Session` is created here and reused across all API calls — this enables connection pooling, which avoids the TCP handshake overhead on every request and meaningfully speeds up large loops.

In [3]:
load_dotenv('../.env')

ONEMAP_EMAIL = os.getenv('ONEMAP_EMAIL')
ONEMAP_PASSWORD = os.getenv('ONEMAP_PASSWORD')

if not ONEMAP_EMAIL or not ONEMAP_PASSWORD:
    raise SystemExit(
        "Credentials missing. Open ../.env and fill in:\n"
        "  ONEMAP_EMAIL=your_email\n"
        "  ONEMAP_PASSWORD=your_password"
    )

print("Credentials loaded successfully.")

Credentials loaded successfully.


In [4]:
# Shared session for connection pooling across all API calls
session = requests.Session()

_token = {"access_token": None, "expiry": 0}

def get_token():
    """Return a valid OneMap access token, refreshing if expired."""
    if _token["access_token"] and time.time() < int(_token["expiry"]) - 60:
        return _token["access_token"]

    resp = session.post(
        "https://www.onemap.gov.sg/api/auth/post/getToken",
        json={"email": ONEMAP_EMAIL, "password": ONEMAP_PASSWORD},
        timeout=10
    )
    resp.raise_for_status()
    data = resp.json()
    _token["access_token"] = data["access_token"]
    _token["expiry"] = int(data["expiry_timestamp"])
    return _token["access_token"]

# Verify authentication works
token = get_token()
expiry_str = time.strftime('%Y-%m-%d %H:%M:%S', time.localtime(_token["expiry"]))
print(f"Authentication successful!")
print(f"Token expires at: {expiry_str}")

Authentication successful!
Token expires at: 2026-03-22 16:56:39


## Phase 3 — Geocode Unique Addresses

Query the OneMap search API to retrieve lat/lon coordinates for each unique address. Results are cached to `data/geocode_cache.json` so the notebook can be safely interrupted and resumed — already-geocoded addresses are skipped. The cache is saved every 50 calls as a checkpoint.

In [5]:
GEOCODE_URL = 'https://www.onemap.gov.sg/api/common/elastic/search'

STREET_ABBREVS = {
    'JLN': 'JALAN',
    'LOR': 'LORONG',
    'BT':  'BUKIT',
    'KG':  'KAMPONG',   # official Singapore spelling is KAMPONG, not KAMPUNG
}

def normalize_street(name: str) -> str:
    """Expand common Malay abbreviations for OneMap search queries."""
    return ' '.join(STREET_ABBREVS.get(token, token) for token in name.split())

# Verify expansions
print("Abbreviation expansion check:")
tests = [
    'JLN RUMAH TINGGI',
    'LOR 1 TOA PAYOH',
    'BT BATOK ST 11',
    'KG ARANG RD',
    'JLN BT MERAH',       # double expansion
    'ANG MO KIO AVE 3',   # no change expected
]
for t in tests:
    print(f"  {t!r:35s} -> {normalize_street(t)!r}")

Abbreviation expansion check:
  'JLN RUMAH TINGGI'                  -> 'JALAN RUMAH TINGGI'
  'LOR 1 TOA PAYOH'                   -> 'LORONG 1 TOA PAYOH'
  'BT BATOK ST 11'                    -> 'BUKIT BATOK ST 11'
  'KG ARANG RD'                       -> 'KAMPONG ARANG RD'
  'JLN BT MERAH'                      -> 'JALAN BUKIT MERAH'
  'ANG MO KIO AVE 3'                  -> 'ANG MO KIO AVE 3'


In [6]:
GEOCODE_CACHE_FILE = '../data/geocode_cache.json'
GEOCODE_URL = 'https://www.onemap.gov.sg/api/common/elastic/search'

# Load existing cache or start fresh
if os.path.exists(GEOCODE_CACHE_FILE):
    with open(GEOCODE_CACHE_FILE, 'r') as f:
        geocode_cache = json.load(f)
    print(f"Loaded existing geocode cache: {len(geocode_cache):,} entries")
else:
    geocode_cache = {}
    print("No existing cache found — starting fresh.")

# Only process addresses not already in cache
to_geocode = [
    (str(row['block']), row['street_name'])
    for _, row in unique_addresses.iterrows()
    if f"{row['block']} {row['street_name']}" not in geocode_cache
]
print(f"Addresses to geocode: {len(to_geocode):,} (skipping {len(geocode_cache):,} cached)")

call_count = 0
for block, street_name in tqdm(to_geocode, desc="Geocoding"):
    key   = f"{block} {street_name}"
    query = f"{block} {normalize_street(street_name)}"   # expanded query, original key
    try:
        resp = session.get(
            GEOCODE_URL,
            params={
                'searchVal': query,
                'returnGeom': 'Y',
                'getAddrDetails': 'Y',
                'pageNum': 1
            },
            headers={'Authorization': f'Bearer {get_token()}'},
            timeout=10
        )
        resp.raise_for_status()
        results = resp.json().get('results', [])
        if results:
            r = results[0]
            geocode_cache[key] = {
                'lat': float(r['LATITUDE']),
                'lon': float(r['LONGITUDE']),
                'postal': r.get('POSTAL', ''),
                'address': r.get('ADDRESS', '')
            }
        else:
            geocode_cache[key] = None
    except requests.RequestException as e:
        print(f"  [ERROR] {key}: {e}")
        geocode_cache[key] = None

    call_count += 1
    time.sleep(0.3)

    if call_count % 50 == 0:
        with open(GEOCODE_CACHE_FILE, 'w') as f:
            json.dump(geocode_cache, f)

# Final save
with open(GEOCODE_CACHE_FILE, 'w') as f:
    json.dump(geocode_cache, f)

success = sum(1 for v in geocode_cache.values() if v is not None)
failed  = sum(1 for v in geocode_cache.values() if v is None)
print(f"\nGeocoding complete.")
print(f"  Successfully geocoded: {success:,}")
print(f"  Failed / no results:   {failed:,}")

Loaded existing geocode cache: 9,660 entries
Addresses to geocode: 0 (skipping 9,660 cached)


Geocoding: 0it [00:00, ?it/s]



Geocoding complete.
  Successfully geocoded: 9,657
  Failed / no results:   3


## Phase 4 — Fetch Nearest Train Station

For each successfully geocoded address, query the OneMap nearby services API to find the nearest public rail station (MRT or LRT). Both MRT lines (NS, EW, CC, DT, NE, TE, CG, CE, JS, JW) and LRT lines (Bukit Panjang BP, Sengkang SW/SE, Punggol PE/PW) are included. If no station is found within 2 km, the search radius is automatically expanded to 5 km. Results are cached to `mrt_cache.json` for resume safety.

In [7]:
# Public rail prefixes — MRT lines and LRT lines (Bukit Panjang BP, Sengkang SW/SE, Punggol PE/PW)
PUBLIC_RAIL_PREFIXES = ("NS", "EW", "CC", "DT", "NE", "TE", "CG", "CE", "JS", "JW",
                        "BP", "SW", "SE", "PE", "PW")

# Opening dates for stations that opened within the dataset window (2017-Q1 onwards).
# Only TE stations need tracking — all other lines (including all LRT lines) were fully
# open before the dataset starts.
# Source: LTA MRT network opening history.
TRAIN_OPENING_DATES = {
    # TE Stage 1 — 31 Jan 2020: Woodlands North, Woodlands, Woodlands South
    'TE1': '2020-01-31', 'TE2': '2020-01-31', 'TE3': '2020-01-31',
    # TE Stage 2 — 28 Aug 2021: Springleaf, Lentor, Mayflower, Bright Hill, Upper Thomson, Caldecott
    'TE4': '2021-08-28', 'TE5': '2021-08-28', 'TE6': '2021-08-28',
    'TE7': '2021-08-28', 'TE8': '2021-08-28', 'TE9': '2021-08-28',
    # TE Stage 3 — 13 Nov 2022: Mount Pleasant → Gardens by the Bay (TE10–TE22)
    'TE10': '2022-11-13', 'TE11': '2022-11-13', 'TE12': '2022-11-13',
    'TE13': '2022-11-13', 'TE14': '2022-11-13', 'TE15': '2022-11-13',
    'TE16': '2022-11-13', 'TE17': '2022-11-13', 'TE18': '2022-11-13',
    'TE19': '2022-11-13', 'TE20': '2022-11-13', 'TE21': '2022-11-13',
    'TE22': '2022-11-13',
    # TE Stage 4 — 23 Jun 2024: Tanjong Rhu → Bayshore (TE23–TE29)
    'TE23': '2024-06-23', 'TE24': '2024-06-23', 'TE25': '2024-06-23',
    'TE26': '2024-06-23', 'TE27': '2024-06-23', 'TE28': '2024-06-23',
    'TE29': '2024-06-23',
    # TE Stage 5 — 19 Nov 2024: Bedok South, Sungei Bedok
    'TE30': '2024-11-19', 'TE31': '2024-11-19',
}

def is_public_rail(station_id: str) -> bool:
    """Return True for any MRT or LRT station."""
    return station_id.upper().startswith(PUBLIC_RAIL_PREFIXES)

def haversine_distance(lat1: float, lon1: float, lat2: float, lon2: float) -> float:
    """Return straight-line distance in kilometres between two lat/lon points."""
    R = 6371.0
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlambda = math.radians(lon2 - lon1)
    a = math.sin(dphi / 2) ** 2 + math.cos(phi1) * math.cos(phi2) * math.sin(dlambda / 2) ** 2
    return R * 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))

_bad_response_logged = 0  # limit noisy debug output

def get_nearest_train(flat_lat: float, flat_lon: float, token: str, radius: int = 2000, top_n: int = 5):
    """Return list of up to top_n nearest MRT/LRT stations sorted by distance, or [] if none found.

    top_n=5 ensures that even when several future TE stations cluster near an address,
    there is still room in the list for an older fallback station (e.g. EW, NS, DT).
    Phase 6 uses this ranked list to pick the nearest station open at transaction time.
    Each item: {'name': str, 'id': str, 'dist_m': float}
    """
    global _bad_response_logged
    for r in [radius, 5000]:
        try:
            resp = session.get(
                'https://www.onemap.gov.sg/api/public/nearbysvc/getNearestMrtStops',
                params={'latitude': flat_lat, 'longitude': flat_lon, 'radius_in_meters': r},
                headers={'Authorization': token},
                timeout=10
            )
            resp.raise_for_status()
            stations = resp.json()
            if not isinstance(stations, list):
                if _bad_response_logged < 3:
                    print(f"  [DEBUG] Unexpected API response (type={type(stations).__name__}): {stations}")
                    _bad_response_logged += 1
                stations = []
            rail_stations = [s for s in stations if is_public_rail(s.get('id', ''))]
            if rail_stations:
                rail_stations_sorted = sorted(
                    rail_stations,
                    key=lambda s: haversine_distance(flat_lat, flat_lon, float(s['lat']), float(s['lon']))
                )[:top_n]
                return [
                    {
                        'name':   s['name'],
                        'id':     s['id'],
                        'dist_m': round(haversine_distance(flat_lat, flat_lon,
                                                           float(s['lat']), float(s['lon'])) * 1000, 1)
                    }
                    for s in rail_stations_sorted
                ]
        except requests.RequestException as e:
            if _bad_response_logged < 3:
                print(f"  [DEBUG] RequestException at r={r}: {e}")
                _bad_response_logged += 1
            return []
        if r == radius:
            continue  # retry with 5000m
        break
    return []

print("Helper functions defined: is_public_rail(), haversine_distance(), get_nearest_train()")
print(f"TRAIN_OPENING_DATES loaded: {len(TRAIN_OPENING_DATES)} TE stations tracked (Stages 1–5)")

Helper functions defined: is_public_rail(), haversine_distance(), get_nearest_train()
TRAIN_OPENING_DATES loaded: 31 TE stations tracked (Stages 1–5)


In [8]:
TRAIN_CACHE_FILE = '../data/train_cache.json'

# Set RETRY_EMPTY = True to re-process addresses that previously returned no result.
# Use this after an interrupted or partially-failed run to fill in the gaps.
RETRY_EMPTY = True

# Load existing cache or start fresh
if os.path.exists(TRAIN_CACHE_FILE):
    with open(TRAIN_CACHE_FILE, 'r') as f:
        train_cache = json.load(f)
    print(f"Loaded existing train cache: {len(train_cache):,} entries")
else:
    train_cache = {}
    print("No existing train cache found — starting fresh.")

# Process addresses not in cache, plus empty-result entries if RETRY_EMPTY is set
to_lookup = [
    (key, val)
    for key, val in geocode_cache.items()
    if val is not None and (key not in train_cache or (RETRY_EMPTY and train_cache.get(key) == []))
]
already_cached = len(geocode_cache) - sum(1 for v in geocode_cache.values() if v is None) - len(to_lookup)
print(f"Addresses to look up: {len(to_lookup):,} (skipping {already_cached:,} cached with results)")

call_count = 0
for key, geo in tqdm(to_lookup, desc="Train lookup"):
    try:
        result = get_nearest_train(geo['lat'], geo['lon'], get_token())
        train_cache[key] = result
    except Exception as e:
        print(f"  [ERROR] {key}: {e}")
        train_cache[key] = []

    call_count += 1
    time.sleep(0.3)

    if call_count % 50 == 0:
        with open(TRAIN_CACHE_FILE, 'w') as f:
            json.dump(train_cache, f)

# Final save
with open(TRAIN_CACHE_FILE, 'w') as f:
    json.dump(train_cache, f)

with_train = sum(1 for v in train_cache.values() if v)
print(f"\nTrain lookup complete.")
print(f"  Addresses with train result: {with_train:,}")
print(f"  Addresses with no result:    {len(train_cache) - with_train:,}")

Loaded existing train cache: 9,657 entries
Addresses to look up: 0 (skipping 9,657 cached with results)


Train lookup: 0it [00:00, ?it/s]



Train lookup complete.
  Addresses with train result: 9,657
  Addresses with no result:    0


## Phase 5 — Flag Failed Geocodes

Collect all addresses where geocoding returned no result and save them to `failed_geocodes.csv` for manual review. These rows will have null lat/lon and train values in the final dataset.

In [9]:
FAILED_FILE = '../data/failed_geocodes.csv'

failed_keys = [key for key, val in geocode_cache.items() if val is None]

failed_rows = []
for key in failed_keys:
    parts = key.split(' ', 1)
    block = parts[0]
    street = parts[1] if len(parts) > 1 else ''
    failed_rows.append({'block': block, 'street_name': street})

failed_df = pd.DataFrame(failed_rows)
failed_df.to_csv(FAILED_FILE, index=False)

print(f"Failed geocodes: {len(failed_df):,}")
print(f"Saved to: {FAILED_FILE}")
if not failed_df.empty:
    print("\nSample failed addresses:")
    print(failed_df.head(10))

Failed geocodes: 3
Saved to: ../data/failed_geocodes.csv

Sample failed addresses:
  block       street_name
0    36  JLN RUMAH TINGGI
1   215   LOR 8 TOA PAYOH
2  116A      JLN TENTERAM


## Phase 6 — Merge and Save

Map the geocoded coordinates and nearest train results back onto every row of the full dataset using block + street_name as the join key. Save the enriched dataset to `hdb_with_train_distances.csv`.

In [10]:
import re

OUTPUT_FILE = '../merged_data/hdb_with_train_distances.csv'

# Load train cache from file if Phase 4 wasn't run in this session
if 'train_cache' not in dir():
    with open('../data/train_cache.json') as f:
        train_cache = json.load(f)
    print(f"Loaded train cache from file: {len(train_cache):,} entries")

def pick_open_train(ranked_stations, transaction_month):
    """From a ranked list, return the nearest station that was open at transaction_month, or None."""
    for station in ranked_stations:
        sid = station['id']
        open_str = TRAIN_OPENING_DATES.get(sid)
        if open_str is None or transaction_month >= pd.Timestamp(open_str):
            return station
    return None

print("Mapping geocode results onto full dataset...")
df['_key'] = df['block'].astype(str) + ' ' + df['street_name']

df['lat'] = df['_key'].map(lambda k: geocode_cache[k]['lat'] if geocode_cache.get(k) else None)
df['lon'] = df['_key'].map(lambda k: geocode_cache[k]['lon'] if geocode_cache.get(k) else None)

print("Picking nearest open train station per transaction...")
df['month'] = pd.to_datetime(df['month'])
df['_chosen'] = df.apply(
    lambda row: pick_open_train(train_cache.get(row['_key'], []), row['month']),
    axis=1
)
df['nearest_train_line']   = df['_chosen'].apply(
    lambda s: re.match(r'^([A-Za-z]+)', s['id']).group(1) if s else None)
df['nearest_train_dist_m'] = df['_chosen'].apply(
    lambda s: s['dist_m'] if s else None)
df['nearest_train_name']   = df['_chosen'].apply(
    lambda s: s['name'] if s else None)
df = df.drop(columns=['_key', '_chosen'])

print(f"Saving to {OUTPUT_FILE}...")
df.to_csv(OUTPUT_FILE, index=False)

has_train     = df['nearest_train_dist_m'].notna().sum()
missing_train = df['nearest_train_dist_m'].isna().sum()
print(f"\n{'='*60}")
print("SUMMARY")
print(f"{'='*60}")
print(f"Total rows in output:          {len(df):,}")
print(f"Rows with train distance:      {has_train:,}")
print(f"Rows missing train distance:   {missing_train:,}")
if has_train > 0:
    print(f"\nNearest train distance (m):")
    print(f"  Min:  {df['nearest_train_dist_m'].min():.1f}")
    print(f"  Mean: {df['nearest_train_dist_m'].mean():.1f}")
    print(f"  Max:  {df['nearest_train_dist_m'].max():.1f}")
    print(f"\nTrain line distribution:")
    print(df['nearest_train_line'].value_counts().to_string())
    print(f"\nSample station names:")
    print(df['nearest_train_name'].dropna().unique()[:10].tolist())
print(f"\nNew columns added: lat, lon, nearest_train_line, nearest_train_dist_m, nearest_train_name")
print(f"\nDone! Output saved to {OUTPUT_FILE}")

Mapping geocode results onto full dataset...
Picking nearest open train station per transaction...
Saving to ../merged_data/hdb_with_train_distances.csv...

SUMMARY
Total rows in output:          197,432
Rows with train distance:      197,165
Rows missing train distance:   267

Nearest train distance (m):
  Min:  22.2
  Mean: 616.6
  Max:  3531.8

Train line distribution:
nearest_train_line
NS    54910
EW    46678
DT    20590
NE    18168
BP    12759
SW    11332
CC     8392
PE     8280
SE     6534
TE     5483
PW     4000
CG       39

Sample station names:
['ANG MO KIO MRT STATION', 'YIO CHU KANG MRT STATION', 'BEDOK RESERVOIR MRT STATION', 'BEDOK NORTH MRT STATION', 'BEDOK MRT STATION', 'TANAH MERAH MRT STATION', 'KAKI BUKIT MRT STATION', 'KEMBANGAN MRT STATION', 'MARYMOUNT MRT STATION', 'BISHAN MRT STATION']

New columns added: lat, lon, nearest_train_line, nearest_train_dist_m, nearest_train_name

Done! Output saved to ../merged_data/hdb_with_train_distances.csv
